# Batoid Pupil Comparison: Hexapod Defocus Configurations

**Author:** Aaron Roodman
**Date Created:** 2026-06-15
**Last Modified:** 2026-06-15
**Status:** In Progress
**Keywords:** batoid, pupil, donut, hexapod, M2, defocus, FAM, ray trace, spot diagram, AOS

## Description

Use **Batoid** + the LSST optical model to study how the giant-donut **pupil** depends on
*how* the defocus is produced, at the field position of our real giant donut on
**R30_S21** (pixel (1167, 2915)).

Method: trace a **dense grid of rays** (a spot diagram, many spots) from a point source
at that field angle through the defocused system to the detector; the 2-D **histogram of
ray positions is the geometric pupil/donut image**.

Comparisons:
1. **8 mm defocus, extra- and intra-focal:** Camera hexapod alone (±8 mm) vs Camera +
   M2 hexapods ±4 mm each (M2 is powered → contributes differently).
2. **FAM mode:** Camera hexapod ±1.5 mm — the intra- and extra-focal pupils used for
   curvature wavefront sensing.

Hexapod pistons are rigid z-shifts of the `LSSTCamera` group and `M2`
(`withGloballyShiftedOptic`; no `batoid_rubin` data download). Models: design
`LSST_r.yaml`, or as-built `Rubin_v3.12_r.yaml` / `Rubin_v3.14_r.yaml`.

**Output:** comparison figures; no files written by default.

**Based on:** the giant-donut study (`wfs_giant_donut_fit.ipynb`); R30_S21 seq 340
(intra_8mm) / 349 (extra_8mm).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-15 | Aaron Roodman | Initial version: batoid spot-diagram pupil, camera-8mm vs camera-4mm+M2-4mm |
| 2026-06-15 | Aaron Roodman | Field angle from the R30_S21 donut pixel; add intra-focal + FAM (±1.5 mm) comparisons |
| 2026-06-15 | Aaron Roodman | FAM: invert the extra donut (x→−x, y→−y) so intra/extra overlay (residual std 2.08→1.18) |
| 2026-06-15 | Aaron Roodman | §7: extract danish's fit pupil (DonutFactory+SingleDonutModel from the ts_wep Instrument, z_fit=0, full off-axis z_ref) and overlay on the batoid donut (giant camera-only 8 mm + FAM 1.5 mm) with difference + radial profiles |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Field Angle of the R30_S21 Donut](#field)
5. [8 mm Defocus: Camera vs Camera+M2 (intra & extra)](#eightmm)
6. [FAM Mode: Intra vs Extra (±1.5 mm)](#fam)
7. [Danish Pupil vs Batoid Spot Diagram](#danish)


<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# LSST optical model: "LSST_r.yaml" (design ~v3.3),
#   "Rubin_v3.12_r.yaml" (as-built, default), "Rubin_v3.14_r.yaml" (as-built).
model_yaml = "Rubin_v3.12_r.yaml"
wavelength = 620e-9              # metres (r-band ~620 nm)

# Field position: the real giant donut on R30_S21. theta is computed from this pixel
# (set theta_x_deg / theta_y_deg directly to override).
detector_name = "R30_S21"
donut_pixel = (1167, 2915)       # (x, y) px
theta_x_deg = None               # None -> from the pixel above
theta_y_deg = None

# Defocus magnitudes (mm)
defocus_mm = 8.0                 # camera-only defocus
m2_split_mm = 4.0                # split: camera + M2 each this much (-> 8 mm total move)
fam_offset_mm = 1.5              # FAM camera-hexapod offset (curvature sensing)

# Ray grid / histogram
n_pupil = 700                    # rays across the pupil grid (n_pupil^2 spots)
hist_npix = 256                  # output donut image size (pixels)
half_mm_8mm = 4.0                # half-width for the 8 mm donut panels (mm)
half_mm_fam = 1.0                # half-width for the FAM donut panels (mm)

cmap = "viridis"


<a id='setup'></a>
## Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import batoid

import lsst.geom
from lsst.afw.cameraGeom import FIELD_ANGLE, PIXELS
from lsst.obs.lsst import LsstCam

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()
camera = LsstCam.getCamera()


<a id='functions'></a>
## Helper Functions

In [ ]:
def field_angle_from_pixel(detector_name, x, y):
    """Field angle (deg) for a detector pixel, from LSST camera geometry.

    Note: gives the correct field *radius*; the absolute orientation vs batoid's
    field-angle frame is approximate (fine for comparing configs at one field point).
    """
    fa = camera[detector_name].getTransform(PIXELS, FIELD_ANGLE).applyForward(
        lsst.geom.Point2D(x, y))
    return float(np.degrees(fa.getX())), float(np.degrees(fa.getY()))


def build_telescope(fiducial, camera_dz_mm, m2_dz_mm):
    """Apply camera- and M2-hexapod z pistons (signed mm) as rigid global z-shifts.
    +z = extra-focal sense, -z = intra-focal (per the chosen convention)."""
    tel = fiducial
    if camera_dz_mm:
        tel = tel.withGloballyShiftedOptic("LSSTCamera", [0, 0, camera_dz_mm * 1e-3])
    if m2_dz_mm:
        tel = tel.withGloballyShiftedOptic("M2", [0, 0, m2_dz_mm * 1e-3])
    return tel


def trace_donut(tel, theta_x_deg, theta_y_deg, wavelength, n_pupil, npix, half_mm):
    """Trace a dense pupil ray grid; histogram (un-vignetted) positions -> donut image,
    centred on the donut centroid. Returns dict(image, span_mm, n_rays)."""
    rays = batoid.RayVector.asGrid(
        optic=tel, wavelength=wavelength,
        theta_x=np.deg2rad(theta_x_deg), theta_y=np.deg2rad(theta_y_deg), nx=n_pupil)
    tel.trace(rays)
    ok = ~rays.vignetted & ~rays.failed
    x = rays.x[ok] * 1e3; y = rays.y[ok] * 1e3
    cx, cy = x.mean(), y.mean()
    H, _, _ = np.histogram2d(x, y, bins=npix,
                             range=[[cx - half_mm, cx + half_mm],
                                    [cy - half_mm, cy + half_mm]])
    return {"image": H.T, "span_mm": float(x.max() - x.min()), "n_rays": int(ok.sum())}


def trace_configs(fiducial, configs, theta, n_pupil, npix, half_mm):
    """configs: list of (label, camera_dz_mm, m2_dz_mm). Returns list of (label, donut)."""
    out = []
    for label, cdz, mdz in configs:
        tel = build_telescope(fiducial, cdz, mdz)
        out.append((label, trace_donut(tel, theta[0], theta[1], wavelength,
                                        n_pupil, npix, half_mm)))
    return out


def azimuthal_profile(image, half_mm, nbin=120):
    """Azimuthally averaged radial profile of a centred donut image -> (r_mm, value)."""
    n = image.shape[0]
    yy, xx = np.mgrid[0:n, 0:n]
    c = (n - 1) / 2.0
    r = np.hypot(xx - c, yy - c) * (2 * half_mm / n)
    edges = np.linspace(0, half_mm, nbin + 1)
    idx = np.digitize(r.ravel(), edges) - 1
    v = image.ravel()
    rc, prof = [], []
    for k in range(nbin):
        sel = idx == k
        if sel.any():
            rc.append(0.5 * (edges[k] + edges[k + 1])); prof.append(v[sel].mean())
    return np.array(rc), np.array(prof)


def show_pair_with_diff(ax_a, ax_b, ax_d, da, db, half_mm, labels):
    """Plot donut A, donut B, and A-B on three axes."""
    vmax = max(da["image"].max(), db["image"].max())
    ext = [-half_mm, half_mm, -half_mm, half_mm]
    for ax, d, lab in ((ax_a, da, labels[0]), (ax_b, db, labels[1])):
        ax.imshow(d["image"], origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
        ax.set_title(f"{lab}\nspan {d['span_mm']:.2f} mm", fontsize=9)
        ax.set_xlabel("mm"); ax.set_ylabel("mm")
    diff = da["image"] - db["image"]
    v = np.nanpercentile(np.abs(diff), 99.5)
    im = ax_d.imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=ext)
    ax_d.set_title(f"({labels[0]}) − ({labels[1]})", fontsize=9)
    ax_d.set_xlabel("mm"); ax_d.set_ylabel("mm")
    return im


<a id='field'></a>
## Field Angle of the R30_S21 Donut

In [ ]:
if theta_x_deg is None or theta_y_deg is None:
    tx, ty = field_angle_from_pixel(detector_name, *donut_pixel)
else:
    tx, ty = theta_x_deg, theta_y_deg
theta = (tx, ty)
print(f"{detector_name} pixel {donut_pixel}: field angle = ({tx:.4f}, {ty:.4f}) deg, "
      f"radius = {np.hypot(tx, ty):.4f} deg")
print(f"model: {model_yaml}")


<a id='eightmm'></a>
## 8 mm Defocus: Camera vs Camera+M2 (intra & extra)

In [ ]:
fiducial = batoid.Optic.fromYaml(model_yaml)

configs_8mm = [
    ("extra: cam +8",            +defocus_mm, 0.0),
    ("extra: cam +4 + M2 +4",    +m2_split_mm, +m2_split_mm),
    ("intra: cam -8",            -defocus_mm, 0.0),
    ("intra: cam -4 + M2 -4",    -m2_split_mm, -m2_split_mm),
]
d8 = dict(trace_configs(fiducial, configs_8mm, theta, n_pupil, hist_npix, half_mm_8mm))
for lab, d in d8.items():
    print(f"  {lab:>24}: span {d['span_mm']:.2f} mm, {d['n_rays']} rays")

# Rows: extra, intra.  Cols: camera-only, camera+M2, difference.
fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
show_pair_with_diff(axes[0, 0], axes[0, 1], axes[0, 2],
                    d8["extra: cam +8"], d8["extra: cam +4 + M2 +4"], half_mm_8mm,
                    ("extra: cam +8", "extra: cam +4 + M2 +4"))
show_pair_with_diff(axes[1, 0], axes[1, 1], axes[1, 2],
                    d8["intra: cam -8"], d8["intra: cam -4 + M2 -4"], half_mm_8mm,
                    ("intra: cam -8", "intra: cam -4 + M2 -4"))
fig.suptitle(f"8 mm defocus pupil — {model_yaml}, field ({tx:.2f}, {ty:.2f})°  "
             f"[camera-only vs camera+M2]", fontsize=12)
plt.show()


In [ ]:
# Azimuthally averaged radial profiles for the 8 mm configs.
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for ax, side in zip(axs, ("extra", "intra")):
    for lab, d in d8.items():
        if lab.startswith(side):
            r, p = azimuthal_profile(d["image"], half_mm_8mm)
            ax.plot(r, p, label=lab)
    ax.set_xlabel("radius from centre [mm]"); ax.set_ylabel("rays / pixel")
    ax.set_title(f"{side}-focal radial profile"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()


<a id='fam'></a>
## FAM Mode: Intra vs Extra (±1.5 mm)

In [ ]:
configs_fam = [
    ("FAM intra: cam -1.5", -fam_offset_mm, 0.0),
    ("FAM extra: cam +1.5", +fam_offset_mm, 0.0),
]
dfam = dict(trace_configs(fiducial, configs_fam, theta, n_pupil, hist_npix, half_mm_fam))
for lab, d in dfam.items():
    print(f"  {lab:>22}: span {d['span_mm']:.2f} mm, {d['n_rays']} rays")

# Intra and extra are related by a parity flip, so invert the extra donut
# (x -> -x, y -> -y, i.e. a 180° point reflection of the centred image) to overlay it
# on the intra donut. The difference then shows the genuine intra/extra mismatch.
extra = dfam["FAM extra: cam +1.5"]
extra_inv = {"image": extra["image"][::-1, ::-1], "span_mm": extra["span_mm"]}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)
show_pair_with_diff(axes[0], axes[1], axes[2],
                    dfam["FAM intra: cam -1.5"], extra_inv, half_mm_fam,
                    ("FAM intra: cam -1.5", "FAM extra: cam +1.5 (inverted x,y)"))
fig.suptitle(f"FAM-mode pupil (camera ±{fam_offset_mm} mm) — {model_yaml}, "
             f"field ({tx:.2f}, {ty:.2f})°  [extra inverted to match intra]", fontsize=11)
plt.show()

# Radial profiles are azimuthal averages (flip-invariant), so compare directly.
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for lab, d in dfam.items():
    r, p = azimuthal_profile(d["image"], half_mm_fam)
    ax.plot(r, p, label=lab)
ax.set_xlabel("radius from centre [mm]"); ax.set_ylabel("rays / pixel")
ax.set_title("FAM intra vs extra radial profile"); ax.grid(alpha=0.3); ax.legend()
plt.show()


<a id='danish'></a>
## Danish Pupil vs Batoid Spot Diagram

Extract the **pupil danish actually uses in its fit** and overlay it on the batoid
spot-diagram donut. danish has no method to return the pupil, so we build its
`DonutFactory` + `SingleDonutModel` from the **ts_wep `Instrument`** (pupil obscuration
`R_outer`/`R_inner`, `mask_params` for M1/M2/M3/L1/Filter/spider, focal length), set
`z_ref` to the instrument **off-axis Zernikes** (which carry the defocus + design
off-axis aberration), use **no fitted Zernikes** (`z_fit=()`) and a tiny `fwhm` (0.3″),
and draw — giving danish's geometric donut/pupil.

Per **Josh Meyers**, danish/ts_wep assume the giant-donut defocus is **camera-only**, so
the danish giant pupil is compared to the batoid **camera-only +8 mm** donut (not the
camera-4 + M2-4 that physically produced the data — that mismatch is the A−B difference
in §5). danish and batoid share the field-angle convention, so the donuts register
directly (identity, no flip). Both images are unit-sum normalised; the difference shows
where danish's parametric pupil departs from batoid's full ray trace (vignetting/baffles).


In [ ]:
import danish
from lsst.ts.wep.utils import getTaskInstrument, DefocalType

compare_npix = 255   # danish requires odd npix; common grid for danish & batoid
danish_fwhm = 0.3    # arcsec; tiny blur -> sharp danish pupil edges


def danish_pupil_donut(detector_name, defocal_mm, defocal_type, theta_deg, npix,
                       half_mm, fwhm=0.3):
    """Geometric donut/pupil danish uses in its fit, at the given defocus, on a ±half_mm
    grid (unit-sum normalised). Builds danish's DonutFactory + SingleDonutModel from the
    ts_wep Instrument; z_ref = off-axis coeffs (carry defocus + design off-axis), and no
    fitted Zernikes (z_fit=())."""
    inst = getTaskInstrument("LSSTCam", detector_name)
    inst.defocalOffset = defocal_mm * 1e-3
    txd, tyd = theta_deg
    z_ref = inst.getOffAxisCoeff(txd, tyd, defocal_type, "r",
                                 nollIndicesModel=np.arange(79),
                                 nollIndicesIntr=np.arange(4, 23))
    pscale = 2 * half_mm * 1e-3 / npix          # m/pixel so the image spans ±half_mm
    factory = danish.DonutFactory(
        R_outer=inst.radius, R_inner=inst.radius * inst.obscuration,
        mask_params=inst.maskParams, focal_length=inst.focalLength, pixel_scale=pscale)
    model = danish.SingleDonutModel(factory, z_ref=z_ref, z_terms=(),
                                    thx=np.deg2rad(txd), thy=np.deg2rad(tyd), npix=npix)
    img = model.model(flux=1.0, dx=0.0, dy=0.0, fwhm=fwhm, z_fit=())
    return img / img.sum()


def batoid_norm(cam_dz_mm, m2_dz_mm, half_mm, npix):
    """Batoid spot-diagram donut on a ±half_mm grid, unit-sum normalised."""
    tel = build_telescope(fiducial, cam_dz_mm, m2_dz_mm)
    b = trace_donut(tel, tx, ty, wavelength, n_pupil, npix, half_mm)
    return b["image"] / b["image"].sum()


# danish assumes CAMERA-ONLY defocus -> compare to batoid camera-only.
comparisons = [
    ("giant: camera-only +8 mm", defocus_mm,   DefocalType.Extra, half_mm_8mm),
    ("FAM: camera +1.5 mm",      fam_offset_mm, DefocalType.Extra, half_mm_fam),
]
results = []
for label, cdz, dtype, half in comparisons:
    B = batoid_norm(cdz, 0.0, half, compare_npix)
    D = danish_pupil_donut(detector_name, cdz, dtype, theta, compare_npix, half, danish_fwhm)
    results.append((label, half, B, D))

fig, axes = plt.subplots(len(results), 3, figsize=(13, 4.6 * len(results)),
                         constrained_layout=True)
axes = np.atleast_2d(axes)
for row, (label, half, B, D) in enumerate(results):
    ext = [-half, half, -half, half]; vmax = max(B.max(), D.max())
    axes[row, 0].imshow(B, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
    axes[row, 0].set_title(f"batoid — {label}", fontsize=9)
    axes[row, 1].imshow(D, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
    axes[row, 1].set_title("danish pupil (z_fit=0)", fontsize=9)
    diff = B - D; v = np.nanpercentile(np.abs(diff), 99.5)
    axes[row, 2].imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=ext)
    axes[row, 2].set_title("batoid − danish", fontsize=9)
    for a in axes[row]:
        a.set_xlabel("mm"); a.set_ylabel("mm")
fig.suptitle(f"Danish pupil vs batoid spot diagram — {model_yaml}, "
             f"field ({tx:.2f}, {ty:.2f})°", fontsize=12)
plt.show()

# Azimuthally averaged radial profiles (orientation-independent).
fig, axs = plt.subplots(1, len(results), figsize=(7 * len(results), 4.4),
                        constrained_layout=True)
axs = np.atleast_1d(axs)
for ax, (label, half, B, D) in zip(axs, results):
    rB, pB = azimuthal_profile(B, half); rD, pD = azimuthal_profile(D, half)
    ax.plot(rB, pB, label="batoid"); ax.plot(rD, pD, label="danish")
    ax.set_title(label); ax.set_xlabel("radius [mm]"); ax.set_ylabel("norm. illumination")
    ax.grid(alpha=0.3); ax.legend()
plt.show()
